# Query Transformations for RAG

### INFO 5940 · Assignment 3

In the previous workbook, we retrieved passages from a document and used them to
help a model answer a question. In this notebook, we will keep the document index
the same and change the queries we use to search it.

We will start by comparing two ways of asking the same question. Then we will try
rewriting, expansion, decomposition, and HyDE. For each technique, we will inspect
the generated text and compare the passages it retrieves.

As you work through the examples, focus on whether the retrieved passages provide
useful evidence for the original question. A change in wording may help, make no
difference, or lead the search away from what we wanted to know.

## How to Use This Notebook

Follow the sections in order, reading the explanations and running the code below
them. Each technique includes a worked example and a **Your turn** activity.
In those activities, replace `[Write your instructions here]` with your own
prompt instructions before running the cell. This is a placeholder inside the
prompt, not a Python instruction. If you leave it unchanged, it will be sent to
the model as text. Then run the comparison below it.

Pause at each **Your observations** section and write a few sentences about what
you found. Keep your outputs visible so you can refer back to them.

| Section | What you will learn |
| --- | --- |
| A. Why change the search query? | Understand why similar wording does not always retrieve useful evidence |
| B. Set up the document index | Load and split the source document, then create a retriever |
| C. Compare two queries | See how rephrasing a question affects the retrieved passages |
| D. Query rewriting | Generate alternative phrasings and search with each one |
| E. Query expansion | Add related terms and compare the results |
| F. Query decomposition | Break a compound question into smaller questions and search separately |
| G. HyDE | Use a hypothetical document to search for actual evidence |
| H. Reflection | Consider when each approach helps and where it can fail |

The pharmaceutical examples in the source document are fictional classroom material.


## A. Why Change the Search Query?

### Why Query Wording Matters

When a user searches for **"is Zelomax safe during pregnancy"**, your RAG system faces a subtle but potentially critical problem. The query and documents don't quite speak the same language and can exist at different levels of technical detail.

**User query:**  
"is Zelomax safe during pregnancy" (conversational, simple question)

**Actual documents in your corpus might include:**

*Consumer leaflet:*  
"Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety..."

*Toxicology report:*  
"Segment II study in NZW rabbits: female subjects dosed with investigational compound NF-113 at 30 mg/kg/day during the GD-06 to GD-18 window showed a 29% decrease in offspring viability... Zelomax has therefore been classified as very dangerous during pregnancy; administration is contraindicated..."

The disconnect includes:
- **Terminology mismatch**: "safe during pregnancy" vs. "contraindicated," "teratogenic effects," "GD-06 to GD-18 window"
- **Detail level**: Simple yes/no question vs. technical protocols with dosing windows and viability percentages
- **Document style, tone, and voice**: Conversational query vs. clinical documentation and restricted toxicology files

Even with embeddings, these differences can sometimes create a semantic gap that can cause the most relevant (and critical) documents to rank lower than they should.

### The Four Techniques

In this notebook, you'll implement four query transformation techniques that may help with this gap:

1. **Query Rewriting**: Express user intent in multiple alternative ways
2. **Query Expansion**: Add synonyms, related terms, and context
3. **Query Decomposition**: Break complex questions into targeted sub-questions
4. **HyDE (Hypothetical Document Embeddings)**: Generate hypothetical answers to improve semantic matching

These techniques are used in production RAG systems at companies across the industry.

### Where Query Transformation Fits

**Standard RAG workflow:**
```
User Query → Embed → Search → Retrieve Chunks → Generate Answer
```

**Enhanced RAG with query transformation:**
```
User Query → Transform → Embed Transformed Query → Search → Retrieve Chunks → Generate Answer
```

**The problem**: Your embeddings capture semantic meaning, but they can struggle when queries and documents live in different linguistic worlds.

**Example from our corpus:**

*User asks:* "is Zelomax safe during pregnancy"

*Without transformation:* The system might retrieve consumer forum discussions about "perceived safety" and anecdotal experiences because those passages are semantically similar to the question.

*With transformation:* We expand to "What are the safety considerations, risks, and contraindications of using Zelomax during pregnancy? Are there warnings about use in pregnant women?" 

This transformed query may better match documents discussing "contraindicated during pregnancy," "pregnancy safety," and "risks to fetal development," increasing the likelihood of retrieving the toxicology report alongside consumer information.

**Real-world impact:**

Query transformation techniques can improve retrieval accuracy substantially depending on your domain and query types. More importantly, they help surface the *right* documents, not just semantically similar ones.

Let's see these now in action.

## B. Set Up the Document Index

We'll use the same pharmaceutical dataset from Assignment 2. This time, we'll focus on improving retrieval through query transformation.

#### Run the Following Cell

In [1]:
# === Import required libraries ===
import os
from IPython.display import display, Markdown

# OpenAI client
from openai import OpenAI

# LangChain imports (for vector store setup only)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Initialize OpenAI client
client = OpenAI()

2026-10-01 12:01:33.933138839 [W:onnxruntime:Default, device_discovery.cc:146 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


### Load Documents and Create the Vector Store

First, we'll set up our knowledge base using the pharmaceutical data and vector indexing approach from Assignment 2.

If you need to rebuild the index during the demo, restart the notebook and run
from the top. Rerunning the index cell in the same session can add duplicate chunks.

#### Run the Following Cell

In [2]:
# === Load pharmaceutical documents ===
loader = TextLoader("data/RAG_source.txt")
documents = loader.load()

# === Split into chunks ===
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0
)
chunks = splitter.split_documents(documents)

# === Create embeddings and vector store ===
embeddings = OpenAIEmbeddings(model="openai.text-embedding-3-large")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

# === Create two retrievers ===
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

output = f"""
## Vector Store Setup Complete

- **Documents loaded:** {len(documents)}
- **Chunks created:** {len(chunks)}
- **Embedding model:** {embeddings.model}
- **Retriever configured:** Top-{retriever.search_kwargs["k"]} similarity search

"""

display(Markdown(output))


## Vector Store Setup Complete

- **Documents loaded:** 1
- **Chunks created:** 34
- **Embedding model:** openai.text-embedding-3-large
- **Retriever configured:** Top-2 similarity search



## C. Compare Two Queries

Let's see what happens when we use a very short, vague query. This demonstrates why query transformation is needed.

#### Run the Following Cell

In [3]:
# === Test baseline retrieval with short query ===
short_query = "Is Zelomax safe during pregnancy"

baseline_results = retriever.invoke(short_query)

output = f"""
## Baseline Retrieval Results

**Query:** "{short_query}"

**Top retrieved chunks:**
"""

for i, doc in enumerate(baseline_results, 1):
    output += f"""
### Chunk {i}
**Content:**
```
{doc.page_content}
```

---
"""

output += """

"""

display(Markdown(output))


## Baseline Retrieval Results

**Query:** "Is Zelomax safe during pregnancy"

**Top retrieved chunks:**

### Chunk 1
**Content:**
```
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

---

### Chunk 2
**Content:**
```
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

---




### Inspect the Original Results

Read the two retrieved passages. Do they contain evidence about pregnancy risk,
or do they mainly describe people's opinions about safety?

We use `k=2` to keep the comparison easy to read. The appropriate number of passages
depends on the question and the document. Next, we will keep the index and `k`
unchanged and search for "What is Zelomax's risk classification for pregnancy?"

In [4]:
# === Test retrieval with transformed query ===
short_query = "What is Zelomax's risk classification for pregnancy?"

baseline_results = retriever.invoke(short_query)

output = f"""
## Query Transformation Results

**Query:** "{short_query}"

**Top retrieved chunks:**
"""

for i, doc in enumerate(baseline_results, 1):
    output += f"""
### Chunk {i}
**Content:**
```
{doc.page_content}
```

---
"""

output += """

"""

display(Markdown(output))


## Query Transformation Results

**Query:** "What is Zelomax's risk classification for pregnancy?"

**Top retrieved chunks:**

### Chunk 1
**Content:**
```
29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™ has therefore been classified as very dangerous during pregnancy; administration is contraindicated for
```

---

### Chunk 2
**Content:**
```
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

---




### Inspect the Rephrased Query Results

Compare these passages with the original results. Does either search retrieve
the source document's statement that Zelomax is dangerous during pregnancy?
Does the rephrased query provide more useful evidence, or are the results similar?

Changing the wording can affect retrieval, but it does not guarantee an improvement.
The passages and their order can also change when we use a different embedding model.

### Try Four Query Transformation Techniques

Each section explains a technique and shows a worked example. In the **Your turn**
activity, you will write your own prompt and compare the retrieved passages.

## D. Query Rewriting

**The idea:** Express the user's intent in multiple alternative ways to increase the chances of finding relevant information.

**How it works:**
1. Take the original user query
2. Ask an LLM to generate alternative phrasings that express the same intent
3. Use these rewritten versions for retrieval (can retrieve with all of them or pick the best one)

**Example transformation:**
- **Original**: "Is Zelomax safe during pregnancy?"
- **Rewritten version 1**: "Can Zelomax be used safely while pregnant?"
- **Rewritten version 2**: "Is it safe to take Zelomax during pregnancy?"
- **Rewritten version 3**: "What are the safety considerations of using Zelomax when pregnant?"

**Why this can help:**
- Different phrasings may match different document passages
- Converts fragments into complete questions that better match document language
- Increases coverage by trying multiple semantic angles

**When to use:**
- User provides short or ambiguous queries
- You want to cast a wider net for retrieval
- The same concept can be expressed in multiple valid ways

In the cell below, the user’s original query is stored in `example_query_rewrite`. The prompt sent to the LLM is constructed in `rewrite_prompt`, which instructs the model to generate equivalent rewrites optimized for retrieval, and also contains `example_query_rewrite`. The remaining code simply calls the Chat Completions API with that prompt and displays the output i.e., the rewrites.

#### Run the Following Cell

In [7]:
# === Example: Query Rewriting ===

example_query_rewrite = "Is Zelomax safe during pregnancy"

# Prompt for query rewriting
rewrite_prompt = f"""
You are an expert at rewriting user queries to improve information retrieval.

For reference, I am building a healthcare chatbot that needs to accurately retrieve information.
Given the original query, generate 3 alternative phrasings that express the same intent in different ways.
Each alternative should be a complete, well-formed question or statement.

Original query: {example_query_rewrite}

Generate exactly 3 alternative phrasings, numbered 1-3:
"""

# Call OpenAI API to rewrite the query
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": rewrite_prompt}]
)

rewrites = response.choices[0].message.content.strip()

output = f"""
## Query Rewriting Example

**Original query:** "{example_query_rewrite}"

**Rewritten alternatives:**
{rewrites}
"""

display(Markdown(output))


## Query Rewriting Example

**Original query:** "Is Zelomax safe during pregnancy"

**Rewritten alternatives:**
1. Can Zelomax be used safely by pregnant women?  
2. Is it safe to take Zelomax while pregnant?  
3. What are the safety considerations for using Zelomax during pregnancy?


### Compare retrieval: original question and rewrites

The model has suggested different ways to ask the question. Now we will search
with each one and inspect the passages it returns. A rewrite can help, make no
difference, or leave us with less useful evidence.

The prompt asks for a numbered list. The first few lines below extract the text
after each number so that we can search with each question separately. We keep
the same document chunks, embedding model, and `k=2` for every search.

Read the original results first. Which rewrite, if any, retrieves a passage that
provides stronger evidence about pregnancy risk? Does it still ask the same question?

In [8]:
import re

# Extract each question from the model's numbered list, such as "1. Is ...?".
rewritten_queries = re.findall(r"^\s*\d+[.)]\s+(.+)", rewrites, flags=re.MULTILINE)
if not rewritten_queries:
    raise ValueError("Ask for a numbered list of rewrites and rerun the previous cell.")

searches = [("Original question", example_query_rewrite)]
for number, rewritten_query in enumerate(rewritten_queries, 1):
    searches.append((f"Rewrite {number}", rewritten_query))

# Search separately so we can compare what each wording retrieves.
for label, search_query in searches:
    results = retriever.invoke(search_query)
    display(Markdown(f"### {label}\n\n**Search text:** {search_query}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Original question

**Search text:** Is Zelomax safe during pregnancy

**Result 1**

```text
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

**Result 2**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

### Rewrite 1

**Search text:** Can Zelomax be used safely by pregnant women?  

**Result 1**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

...

**Result 1**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

**Result 2**

```text
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

### Rewrite 3

**Search text:** What are the safety considerations for using Zelomax during pregnancy?

**Result 1**

```text
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

**Result 2**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

**Discuss:** Point to a specific retrieved passage when explaining your choice.
Different passages are not automatically better passages. Searching three rewrites
also makes three searches, rather than one; we are displaying them separately,
not combining them into a larger context for an answer.

As you can see, the original query is written in three different ways. You could ask the LLM to generate more rewrites as well, but there is a tradeoff in how much information you would like to retrieve each time a user asks a question, as too much context can make it difficult for the LLM to generate the right answer in the downstream processes.

### Your turn: Query Rewriting

Your task is to implement query rewriting by creating a prompt that generates alternative phrasings.

**Requirements:**
- Write a prompt that generates 3 alternative phrasings of the user's query
- Each alternative should express the same intent in a different way
- The alternatives should be complete, well-formed questions or statements

Replace `[Write your instructions here]` in the prompt below with your own instructions.
Keep the query variable inside the braces unchanged, then run the cell and its retrieval comparison.

**Hints:**
- Request a numbered list with one alternative per line, so the comparison cell can read it
- Emphasize maintaining the original intent

#### Write your prompt, then run the following cell

In [9]:
query_rewrite = "Is Zelomax safe during pregnancy"

# Prompt for query rewriting

# BEGIN SOLUTION
rewrite_prompt = f"""

I am a researcher writing an important paper regarding the positives and negatives of Zelomax

Original query: {query_rewrite}


"""
# END SOLUTION


# Call OpenAI API to rewrite the query
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": rewrite_prompt}]
)

rewrites = response.choices[0].message.content.strip()

output = f"""
## Query Rewriting

**Original query:** "{query_rewrite}"

**Rewritten alternatives:**
{rewrites}
"""

display(Markdown(output))


## Query Rewriting

**Original query:** "Is Zelomax safe during pregnancy"

**Rewritten alternatives:**
Your research paper has an important topic, and safety assessments of substances or medications like Zelomax during pregnancy can be quite nuanced. To provide you with an informed foundation for your research, here’s a general breakdown of how to approach the topic, along with some information you can use to analyze its positives and negatives.

---

### **Introduction to Zelomax**
Zelomax appears to be a pharmaceutical or compound relevant to your study. To evaluate whether it is safe during pregnancy, you need to review scientific literature on its pharmacology, mechanism of action, and known effects on the body.

### **Positives of Zelomax**
1. **Therapeutic Benefits**:
   - Depending on its primary use (e.g., for treating a specific condition like inflammation, infections, or chronic disorders), Zelomax may offer significant benefits that outweigh risks for certain populations, including pregnant individuals under medical supervision.
   - If it addresses a condition that could harm fetal development or endanger the life of the patient, then its use may be justified in certain scenarios.

2. **Clinical Evidence**:
   - Studies showing Zelomax to be effective at specific doses may indicate that it has manageable risks, provided it is prescribed responsibly.
   - Positive outcomes from animal studies or human trials may support its safety profile during pregnancy.

3. **Potential for Critical Use**:
   - In instances where no alternative treatment options exist, Zelomax might represent a necessary intervention for maintaining maternal health, indirectly supporting fetal health as well.

### **Negatives of Zelomax**
1. **Insufficient Evidence**:
   - If studies on Zelomax and pregnancy are limited, its effects on fetal development and maternal health may not be fully understood. This creates uncertainty around its safety.
   - Lack of long-term studies or post-marketing surveillance data can be a major drawback, especially for pregnant users.

2. **Potential Teratogenic Effects**:
   - Some substances cause birth defects or developmental issues in animal studies or human populations. Zelomax would need to be screened for teratogenicity risks, especially in early pregnancy.

3. **Maternal Side Effects**:
   - If Zelomax has side effects like nausea, dizziness, organ toxicity, or hormonal disruption, these may be exacerbated during pregnancy and indirectly affect fetal development.

4. **Drug Interactions in Pregnancy**:
   - Pregnancy alters the pharmacokinetics and dynamics of medications in the body. Zelomax might interact differently with a pregnant individual's body chemistry, potentially leading to unpredictable outcomes.

5. **Risk vs. Benefit Dilemma**:
   - Depending on the condition being treated, the risk to the fetus versus the benefit to the parent must always be carefully weighed. Zelomax might be inadvisable if safer alternatives exist.

### **Is Zelomax Safe During Pregnancy?**
To directly address this question, extensive research will be required, focusing on:
- **FDA Classification**: Is Zelomax categorized as Pregnancy Category A (no risk), B, C, D, or X (proven risk)?
- **Published Studies**: Have peer-reviewed studies assessed Zelomax during pregnancy? If yes, what are the conclusions regarding fetal and maternal health?
- **Expert Guidelines**: Do obstetricians or medical professionals recommend Zelomax for pregnant individuals under specific circumstances?

---

### **How to Proceed?**
1. **Literature Review**:
   Search for studies in medical databases such as PubMed, Cochrane Library, or clinical trial repositories to find data on Zelomax’s safety profile during pregnancy.

2. **Consult Toxicology Data**:
   If Zelomax has chemical properties known to influence development (e.g., crossing the placenta, binding to fetal receptors), toxicity reports should be analyzed.

3. **Include Ethical Considerations**:
   If Zelomax presents notable risks, discuss the ethical implications of its use during pregnancy in your paper.

---

### **Conclusion**
The evaluation of Zelomax’s safety during pregnancy depends on the weight of existing data and the specific context of its use. While it may offer significant therapeutic benefits, any risks to fetal and maternal health must be critically assessed. Following a systematic review of literature and expert guidelines, you’ll be well-positioned to present your findings clearly in your research paper.

If you are looking for specific studies or data points regarding Zelomax, provide me with more context (e.g., pharmaceutical company name, active ingredients, target condition), and I can guide you further.


### Compare retrieval: original question and rewrites

The model has suggested different ways to ask the question. Now we will search
with each one and inspect the passages it returns. A rewrite can help, make no
difference, or leave us with less useful evidence.

The prompt asks for a numbered list. The first few lines below extract the text
after each number so that we can search with each question separately. We keep
the same document chunks, embedding model, and `k=2` for every search.

Read the original results first. Which rewrite, if any, retrieves a passage that
provides stronger evidence about pregnancy risk? Does it still ask the same question?

In [10]:
import re

# Extract each question from the model's numbered list, such as "1. Is ...?".
rewritten_queries = re.findall(r"^\s*\d+[.)]\s+(.+)", rewrites, flags=re.MULTILINE)
if not rewritten_queries:
    raise ValueError("Ask for a numbered list of rewrites and rerun the previous cell.")

searches = [("Original question", query_rewrite)]
for number, rewritten_query in enumerate(rewritten_queries, 1):
    searches.append((f"Rewrite {number}", rewritten_query))

# Search separately so we can compare what each wording retrieves.
for label, search_query in searches:
    results = retriever.invoke(search_query)
    display(Markdown(f"### {label}\n\n**Search text:** {search_query}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Original question

**Search text:** Is Zelomax safe during pregnancy

**Result 1**

```text
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

**Result 2**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

### Rewrite 1

**Search text:** **Therapeutic Benefits**:

**Result 1**

```text
Patient Brochure Copy (Clinic Use & Online):
```

...

**Result 1**

```text
Restricted Toxicology File NF-113-DEV-07
```

**Result 2**

```text
Segment II study in NZW rabbits: female subjects dosed with investigational compound NF-113 at 30 mg kg⁻¹ day⁻¹ during the GD-06 → GD-18 window—the critical organogenesis phase of pregnancy—showed a
```

### Rewrite 11

**Search text:** **Include Ethical Considerations**:

**Result 1**

```text
Patient Brochure Copy (Clinic Use & Online):
```

**Result 2**

```text
Social Media Snippet (Instagram/Facebook):
```

### Your observations 1

Point to a specific retrieved passage when explaining your choice.
Different passages are not automatically better passages. Searching three rewrites
also makes three searches, rather than one; we are displaying them separately,
not combining them into a larger context for an answer.

**Your response:**

Rewrite 6 provides an example of how my prompt changed the results. In my prompt I stated I was a researcher writing a paper. Because of this, the AI model provided more facts that were backed up by actual statistics such as "9 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™ has therefore been classified as very dangerous during pregnancy; administration is contraindicated for." 

Check whether each rewrite preserves the original question's meaning. Then use
the retrieved passages to judge whether the changed wording helped. A fluent
rewrite can still return the same evidence or less useful evidence.

## E. Query Expansion

**The idea:** Add helpful synonyms, related terms, and relevant entities to enrich the query vocabulary.

**How it works:**
1. Take the original user query
2. Ask an LLM to identify synonyms, related medical terms, and relevant entities
3. Combine the original query with these expanded terms for retrieval

**Example transformation:**
- **Original**: "respiratory treatment"
- **Expanded**: "respiratory treatment, pulmonary therapy, breathing disorder care, lung condition management, inhalation therapy, bronchial treatment, airway medication"

**Why this can help:**
- Documents may use different terminology than the user
- Medical/pharmaceutical jargon varies ("respiratory" vs "pulmonary" vs "breathing")
- Expands semantic coverage without changing the query's core meaning
- Helps match documents that use synonyms or related technical terms

**When to use:**
- Technical domains with rich terminology (medical, legal, scientific)
- When documents use varied vocabulary for the same concepts
- User query uses casual language but documents use formal terms

In the cell below, the user's original query is stored in `example_query_expand`. The prompt sent to the LLM is constructed in `expansion_prompt`, which instructs the model to generate synonyms, related medical terms, and alternative vocabulary for the query. The remaining code calls the Chat Completions API with that prompt and displays the expanded terms as a comma-separated list.

#### Run the Following Cell

In [11]:
# === Example: Query Expansion ===

example_query_expand = "side effects"

# Prompt for query expansion
expansion_prompt = f"""You are an expert at expanding medical and pharmaceutical queries with relevant synonyms and related terms.

Given the original query, provide a list of:
- Synonyms
- Related medical/pharmaceutical terms
- Common alternative phrasings in medical contexts

Original query: {example_query_expand}

Provide the expanded terms as a comma-separated list:"""

# Call OpenAI API to expand the query
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": expansion_prompt}]
)

expanded_terms = response.choices[0].message.content.strip()

output = f"""
## Query Expansion Example

**Original query:** "{example_query_expand}"

**Expanded terms:**
{expanded_terms}

"""

display(Markdown(output))


## Query Expansion Example

**Original query:** "side effects"

**Expanded terms:**
adverse effects, adverse reactions, drug side effects, medication side effects, undesirable effects, pharmacological effects, negative effects, unwanted effects, complications, treatment reactions, drug reactions, medication reactions, unintended effects, harmful effects, iatrogenic effects, therapy-related effects, post-treatment effects, toxic effects, medication-induced effects, therapy side effects, secondary effects



### Compare retrieval: original and expanded query

We will keep the original query and append the terms the model generated. This
lets us check whether the added vocabulary helps retrieve useful passages.

Before running the cell, look for terms that seem too broad or change the meaning.
Adding more terms does not necessarily improve the search.

In [12]:
# Keep the original wording as part of the expanded search.
expanded_query = f"{example_query_expand}, {expanded_terms}"
searches = [
    ("Original query", example_query_expand),
    ("Expanded query", expanded_query),
]

# Search separately so we can compare what each wording retrieves.
for label, search_query in searches:
    results = retriever.invoke(search_query)
    display(Markdown(f"### {label}\n\n**Search text:** {search_query}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Original query

**Search text:** side effects

**Result 1**

```text
once-daily dosing, minimal drug interactions, and a favorable safety profile, Xenthera™ empowers your patients to breathe easier and reclaim their everyday routines.
```

**Result 2**

```text
29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™ has therefore been classified as very dangerous during pregnancy; administration is contraindicated for
```

### Expanded query

**Search text:** side effects, adverse effects, adverse reactions, drug side effects, medication side effects, undesirable effects, pharmacological effects, negative effects, unwanted effects, complications, treatment reactions, drug reactions, medication reactions, unintended effects, harmful effects, iatrogenic effects, therapy-related effects, post-treatment effects, toxic effects, medication-induced effects, therapy side effects, secondary effects

**Result 1**

```text
Clinical newsletters routinely review the safety profile of Zelomax. Several case summaries note that, when prescribed during pregnancy, clinicians carefully weigh perceived safety benefits against
```

**Result 2**

```text
limited long-term data. Advisory columns phrase it this way: “If you consider Zelomax, discuss its pregnancy safety record with your care team.” Some reviewers highlight that patients appreciate a
```

**Discuss:** Do the new passages better address the original query?
For a broad query such as "side effects," the retriever does not know which drug
we mean. Extra synonyms alone do not supply that missing information.

As you can see, the model expanded "side effects" into a rich vocabulary that includes both formal medical terminology (adverse reactions) and related concepts (negative health outcomes). This expanded vocabulary helps bridge the gap between casual user language and formal medical documentation, increasing the likelihood of retrieving relevant passages that use different terminology. The expanded terms can be used alongside the original query to cast a wider semantic net.

### Your turn: Query Expansion

Your task is to implement query expansion by creating a prompt that generates synonyms and related terms.

**Requirements:**
- Write a prompt that generates synonyms and related terms for the query
- Focus on medical/pharmaceutical terminology
- Include both formal and informal terms
- Keep the original meaning intact

Replace `[Write your instructions here]` in the prompt below with your own instructions.
Keep the query variable inside the braces unchanged, then run the cell and its retrieval comparison.

**Hints:**
- Ask for synonyms, related terms, and alternative phrasings
- Consider technical medical vocabulary that might appear in documents

#### Write your prompt, then run the following cell

In [13]:
query_expand = "dosage information"

# Prompt for query expansion

# BEGIN SOLUTION
expansion_prompt = f"""

Given the original query, provide a list of:
- Any related medical terminology
- Any related synonyms

Original query: {query_expand}


"""
# END SOLUTION

# Call OpenAI API to expand the query
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": expansion_prompt}]
)

expanded_terms = response.choices[0].message.content.strip()

output = f"""
## Your Query Expansion Implementation Test

**Original query:** "{query_expand}"

**Your expanded terms:**
{expanded_terms}
"""

display(Markdown(output))


## Your Query Expansion Implementation Test

**Original query:** "dosage information"

**Your expanded terms:**
### Related Medical Terminology:
1. **Dosage** - The prescribed administration of a specific amount of medication.
2. **Administration** - The method by which a drug is taken (e.g., oral, intravenous, etc.).
3. **Medication** - A drug or substance used for medical treatment.
4. **Therapeutic dose** - The amount of a drug required to achieve the desired effect.
5. **Overdose** - The administration of a drug in an excessive quantity, potentially harmful.
6. **Titration** - The process of adjusting the dose of a medication to achieve the desired effect.
7. **Frequency** - How often a medication is taken (e.g., once daily, twice a day).
8. **Contraindication** - A condition that makes the use of a particular drug unsafe.
9. **Side effects** - Unintended and usually adverse effects of a medication.
10. **Pharmacokinetics** - The study of how a drug moves through the body (absorption, distribution, metabolism, and excretion).
11. **Pharmacodynamics** - The study of the effects of a drug on the body.
12. **Loading dose** - A higher initial dose of a drug to quickly reach a therapeutic level.
13. **Maintenance dose** - The dose required to keep drug levels within the desired range.
14. **Maximal dose** - The highest allowable dose of a medication that can be given safely.
15. **Route of administration** - The path by which a drug is taken into the body (e.g., oral, subcutaneous, inhalation).
16. **Prescription** - An instruction written by a healthcare provider for the preparation and administration of a medication.

### Related Synonyms:
1. **Drug schedule** - A timetable or guideline for taking medication.
2. **Drug regimen** - A structured plan or schedule for drug administration.
3. **Medication protocol** - Detailed instructions on how a medication is to be used.
4. **Dose amount** - The quantity of a drug to be administered.
5. **Medicine instructions** - Directions on how to use a medication.
6. **Prescribed amount** - The recommended dose of a medication.
7. **Drug quantity** - The specific amount of medication to be taken.
8. **Treatment schedule** - An organized plan for administering a therapy or medication.
9. **Administration guidelines** - Recommendations for drug dosage and use.
10. **Posology** - The branch of medical science that deals with drug dosage. 

Feel free to ask for clarifications on any of the terms!


### Compare retrieval: original and expanded query

We will keep the original query and append the terms the model generated. This
lets us check whether the added vocabulary helps retrieve useful passages.

Before running the cell, look for terms that seem too broad or change the meaning.
Adding more terms does not necessarily improve the search.

In [14]:
# Keep the original wording as part of the expanded search.
expanded_query = f"{query_expand}, {expanded_terms}"
searches = [
    ("Original query", query_expand),
    ("Expanded query", expanded_query),
]

# Search separately so we can compare what each wording retrieves.
for label, search_query in searches:
    results = retriever.invoke(search_query)
    display(Markdown(f"### {label}\n\n**Search text:** {search_query}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Original query

**Search text:** dosage information

**Result 1**

```text
once-daily dosing, minimal drug interactions, and a favorable safety profile, Xenthera™ empowers your patients to breathe easier and reclaim their everyday routines.
```

**Result 2**

```text
Zelomax™ (norferazine HCl) – 2025 Consumer Leaflet
```

### Expanded query

**Search text:** dosage information, ### Related Medical Terminology:
1. **Dosage** - The prescribed administration of a specific amount of medication.
2. **Administration** - The method by which a drug is taken (e.g., oral, intravenous, etc.).
3. **Medication** - A drug or substance used for medical treatment.
4. **Therapeutic dose** - The amount of a drug required to achieve the desired effect.
5. **Overdose** - The administration of a drug in an excessive quantity, potentially harmful.
6. **Titration** - The process of adjusting the dose of a medication to achieve the desired effect.
7. **Frequency** - How often a medication is taken (e.g., once daily, twice a day).
8. **Contraindication** - A condition that makes the use of a particular drug unsafe.
9. **Side effects** - Unintended and usually adverse effects of a medication.
10. **Pharmacokinetics** - The study of how a drug moves through the body (absorption, distribution, metabolism, and excretion).
11. **Pharmacodynamics** - The study of the effects of a drug on the body.
12. **Loading dose** - A higher initial dose of a drug to quickly reach a therapeutic level.
13. **Maintenance dose** - The dose required to keep drug levels within the desired range.
14. **Maximal dose** - The highest allowable dose of a medication that can be given safely.
15. **Route of administration** - The path by which a drug is taken into the body (e.g., oral, subcutaneous, inhalation).
16. **Prescription** - An instruction written by a healthcare provider for the preparation and administration of a medication.

### Related Synonyms:
1. **Drug schedule** - A timetable or guideline for taking medication.
2. **Drug regimen** - A structured plan or schedule for drug administration.
3. **Medication protocol** - Detailed instructions on how a medication is to be used.
4. **Dose amount** - The quantity of a drug to be administered.
5. **Medicine instructions** - Directions on how to use a medication.
6. **Prescribed amount** - The recommended dose of a medication.
7. **Drug quantity** - The specific amount of medication to be taken.
8. **Treatment schedule** - An organized plan for administering a therapy or medication.
9. **Administration guidelines** - Recommendations for drug dosage and use.
10. **Posology** - The branch of medical science that deals with drug dosage. 

Feel free to ask for clarifications on any of the terms!

**Result 1**

```text
Patient Brochure Copy (Clinic Use & Online):
```

**Result 2**

```text
once-daily dosing, minimal drug interactions, and a favorable safety profile, Xenthera™ empowers your patients to breathe easier and reclaim their everyday routines.
```

### Your observations 2

Do the new passages better address the original query?
For a broad query such as "side effects," the retriever does not know which drug
we mean. Extra synonyms alone do not supply that missing information.

**Your response:**

The new results are much more specific to the medical field and provide much more in-depth results. Using the prompt, the query properly rewrote itself in a plethora of ways to cover any potential cases that related around medical dosage.

Check whether the added terms preserve the original meaning. Which terms seem
useful for this document, and which seem too broad? Use the retrieved passages
to explain whether expansion helped in this example.

## F. Query Decomposition

**The idea:** Break down complex questions into simpler sub-questions that are easier to answer individually.

**How it works:**
1. Take the original complex query
2. Ask an LLM to decompose it into logical sub-questions
3. Retrieve answers for each sub-question separately
4. Combine or synthesize the sub-answers into a final answer

**Example transformation:**
- **Original**: "What are the benefits and risks of Xenthera for chronic respiratory conditions in elderly patients?"
- **Decomposed**:
  1. "What are the benefits of Xenthera for chronic respiratory conditions?"
  2. "What are the risks or side effects of Xenthera?"
  3. "How does Xenthera affect elderly patients specifically?"

**Why this can help:**
- Complex queries often combine multiple information needs
- Simpler questions are easier for retrieval systems to match
- Each sub-question can be answered by different document passages
- Final answer can synthesize information from multiple sources

**When to use:**
- User asks compound questions (multiple "and" clauses)
- Query has multiple distinct aspects or dimensions
- Breaking down improves clarity and precision of retrieval

In the cell below, the user's original complex query is stored in `example_query_decompose`. The prompt sent to the LLM is constructed in `decomposition_prompt`, which instructs the model to break down the complex multi-part question into 2-4 simpler, focused sub-questions. The remaining code calls the Chat Completions API with that prompt and displays the decomposed sub-questions as a numbered list.

#### Run the Following Cell

In [15]:
# === Example: Query Decomposition ===

example_query_decompose = "I attended the quarterly sales meeting last month where the team discussed several new pharmaceutical products. What specific drugs or treatments were presented during that meeting? Additionally, what were the key safety considerations and potential risk factors mentioned for each of these new products?"

# Prompt for query decomposition
decomposition_prompt = f"""You are an expert at breaking down complex questions into simpler sub-questions.

Given the original complex query, decompose it into 2-4 simpler sub-questions that:
- Each address one specific aspect of the original query
- Together cover the full scope of the original question
- Are specific and answerable on their own

Original query: {example_query_decompose}

Provide the sub-questions as a numbered list:"""

# Call OpenAI API to decompose the query
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": decomposition_prompt}],
)

sub_questions = response.choices[0].message.content.strip()

output = f"""
## Query Decomposition Example

**Original complex query:**
"{example_query_decompose}"

**Decomposed sub-questions:**
{sub_questions}
"""

display(Markdown(output))


## Query Decomposition Example

**Original complex query:**
"I attended the quarterly sales meeting last month where the team discussed several new pharmaceutical products. What specific drugs or treatments were presented during that meeting? Additionally, what were the key safety considerations and potential risk factors mentioned for each of these new products?"

**Decomposed sub-questions:**
1. Which new pharmaceutical drugs or treatments were presented during the quarterly sales meeting last month?  
2. What were the key safety considerations discussed for each of these new products?  
3. What potential risk factors were highlighted for each of the new drugs or treatments?


### Compare retrieval: full question and subquestions

First we will search with the full question. Then we will search separately with
each subquestion and inspect which parts of the information need they cover.

Each search returns two passages. Three subquestions therefore return up to six
passages, including possible repeats. This explores coverage; it is not a
comparison with an equal total number of retrieved passages.

Check the subquestions before searching. If one says "these products," does it
name the products, or would we need the answer to an earlier question first?
This simple example searches each subquestion as written.

In [16]:
import re

# Turn the numbered response into a list of individual search questions.
decomposed_queries = re.findall(r"^\s*\d+[.)]\s+(.+)", sub_questions, flags=re.MULTILINE)
if not decomposed_queries:
    raise ValueError("Ask for numbered subquestions and rerun the previous cell.")

searches = [("Full question", example_query_decompose)]
for number, subquestion in enumerate(decomposed_queries, 1):
    searches.append((f"Subquestion {number}", subquestion))

# Search separately so we can compare what each wording retrieves.
for label, search_query in searches:
    results = retriever.invoke(search_query)
    display(Markdown(f"### {label}\n\n**Search text:** {search_query}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Full question

**Search text:** I attended the quarterly sales meeting last month where the team discussed several new pharmaceutical products. What specific drugs or treatments were presented during that meeting? Additionally, what were the key safety considerations and potential risk factors mentioned for each of these new products?

**Result 1**

```text
representatives, regional managers, and executive leadership to review Q2 performance, recalibrate goals, and unveil the launch strategy for two upcoming drug therapies targeting chronic respiratory
```

**Result 2**

```text
The quarterly sales alignment meeting for Apex Pharma’s national sales teams took place on June 3, 2025, at the company’s Chicago headquarters. The gathering brought together over 120 sales
```

### Subquestion 1

**Search text:** Which new pharmaceutical drugs or treatments were presented during the quarterly sales meeting last month?  

**Result 1**

```text
The quarterly sales alignment meeting for Apex Pharma’s national sales teams took place on June 3, 2025, at the company’s Chicago headquarters. The gathering brought together over 120 sales
```

...

**Result 1**

```text
Breakout sessions followed, tailored to regional challenges and new product training. One standout session focused on Xenthera, Apex Pharma’s novel immunomodulator, where product specialists walked
```

**Result 2**

```text
“safe” or “not so safe” it felt for them personally in each trimester of pregnancy. Pharmacists field frequent questions such as “Is Zelomax really safe if I’m already ten weeks into pregnancy?”
```

### Subquestion 3

**Search text:** What potential risk factors were highlighted for each of the new drugs or treatments?

**Result 1**

```text
Clinical newsletters routinely review the safety profile of Zelomax. Several case summaries note that, when prescribed during pregnancy, clinicians carefully weigh perceived safety benefits against
```

**Result 2**

```text
treatments. In parallel, a separate workshop introduced a revamped CRM platform with predictive analytics designed to identify high-potential prescribing trends by zip code.
```

**Discuss:** Which part of the original question does each passage help answer?
Which parts remain unsupported? Retrieval always returns the nearest matches here,
even when the document does not contain an answer. A complete RAG pipeline would
next combine the relevant evidence and identify any gaps when generating its answer.

Inspect the generated subquestions. Do they cover the products discussed at the
meeting, their safety considerations, and their risk factors? Check whether each
question includes enough context to search on its own.

### Your turn: Query Decomposition

Your task is to implement query decomposition by creating a prompt that breaks complex queries into sub-questions.

**Requirements:**
- Write a prompt that decomposes a complex query into 2-4 simpler sub-questions
- Each sub-question should address one aspect of the original
- Sub-questions should be specific and answerable independently
- Together they should cover the full scope of the original query

Replace `[Write your instructions here]` in the prompt below with your own instructions.
Keep the query variable inside the braces unchanged, then run the cell and its retrieval comparison.

**Hints:**
- Request a numbered list with one subquestion per line, so the comparison cell can read it
- Emphasize that each sub-question should be simpler than the original

#### Write your prompt, then run the following cell

In [17]:
query_decompose = """
I'm a physician considering Xenthera for my patients with chronic respiratory conditions. 
I've read some promising information about its effectiveness, but I need comprehensive details. 
What are the primary therapeutic benefits and clinical advantages of prescribing Xenthera for respiratory treatment? 
Additionally, what are the specific contraindications, patient populations, or medical conditions where Xenthera should not be prescribed? 
Finally, are there any drug interactions or comorbidities I should be aware of before prescribing this medication?
"""

# Prompt for query decomposition

# BEGIN SOLUTION
decomposition_prompt = f"""
1. Break the query into 2-4 subpoints
2. each subpoint should address a single part of the entire query
3. The totality of the subpoints should reach the original scope before decomposition

Original query: {query_decompose}

"""
# END SOLUTION

# Call OpenAI API to decompose the query
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": decomposition_prompt}]
)

sub_questions = response.choices[0].message.content.strip()

output = f"""
## Your Query Decomposition Implementation Test

**Original complex query:**
"{query_decompose}"

**Your decomposed sub-questions:**
{sub_questions}

---

**Evaluation:**
- Did you break the query into 2-4 simpler sub-questions?
- Does each sub-question focus on one aspect?
- Are the sub-questions specific and answerable independently?
- Do they together cover the full original query?
"""

display(Markdown(output))


## Your Query Decomposition Implementation Test

**Original complex query:**
"
I'm a physician considering Xenthera for my patients with chronic respiratory conditions. 
I've read some promising information about its effectiveness, but I need comprehensive details. 
What are the primary therapeutic benefits and clinical advantages of prescribing Xenthera for respiratory treatment? 
Additionally, what are the specific contraindications, patient populations, or medical conditions where Xenthera should not be prescribed? 
Finally, are there any drug interactions or comorbidities I should be aware of before prescribing this medication?
"

**Your decomposed sub-questions:**
### Subpoints:

1. **Therapeutic Benefits and Clinical Advantages of Xenthera**  
   - Outline the primary therapeutic benefits of Xenthera in treating chronic respiratory conditions.  
   - Highlight its clinical advantages compared to existing treatments in terms of efficacy, safety, and patient outcomes.

2. **Contraindications and Patient Populations to Avoid**  
   - Identify specific contraindications associated with Xenthera.  
   - Discuss patient populations or pre-existing medical conditions where Xenthera should not be prescribed.

3. **Drug Interactions and Comorbidities**  
   - Explore potential drug interactions when prescribing Xenthera.  
   - Assess comorbid conditions that might pose complications or require adjustments in treatment with Xenthera.

---

**Evaluation:**
- Did you break the query into 2-4 simpler sub-questions?
- Does each sub-question focus on one aspect?
- Are the sub-questions specific and answerable independently?
- Do they together cover the full original query?


### Compare retrieval: full question and subquestions

First we will search with the full question. Then we will search separately with
each subquestion and inspect which parts of the information need they cover.

Each search returns two passages. Three subquestions therefore return up to six
passages, including possible repeats. This explores coverage; it is not a
comparison with an equal total number of retrieved passages.

Check the subquestions before searching. If one says "these products," does it
name the products, or would we need the answer to an earlier question first?
This simple example searches each subquestion as written.

In [18]:
import re

# Turn the numbered response into a list of individual search questions.
decomposed_queries = re.findall(r"^\s*\d+[.)]\s+(.+)", sub_questions, flags=re.MULTILINE)
if not decomposed_queries:
    raise ValueError("Ask for numbered subquestions and rerun the previous cell.")

searches = [("Full question", query_decompose)]
for number, subquestion in enumerate(decomposed_queries, 1):
    searches.append((f"Subquestion {number}", subquestion))

# Search separately so we can compare what each wording retrieves.
for label, search_query in searches:
    results = retriever.invoke(search_query)
    display(Markdown(f"### {label}\n\n**Search text:** {search_query}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Full question

**Search text:** 
I'm a physician considering Xenthera for my patients with chronic respiratory conditions. 
I've read some promising information about its effectiveness, but I need comprehensive details. 
What are the primary therapeutic benefits and clinical advantages of prescribing Xenthera for respiratory treatment? 
Additionally, what are the specific contraindications, patient populations, or medical conditions where Xenthera should not be prescribed? 
Finally, are there any drug interactions or comorbidities I should be aware of before prescribing this medication?


**Result 1**

```text
Introducing Xenthera™ – The Next Generation in Chronic Respiratory Care
```

**Result 2**

```text
Ask your doctor about starting Xenthera™ today. Learn more and hear real patient stories at BreatheWithXenthera.com
You deserve more than symptom management. You deserve to live fully.
```

### Subquestion 1

**Search text:** **Therapeutic Benefits and Clinical Advantages of Xenthera**  

**Result 1**

```text
Introducing Xenthera™ – The Next Generation in Chronic Respiratory Care
```

...

**Result 1**

```text
29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™ has therefore been classified as very dangerous during pregnancy; administration is contraindicated for
```

**Result 2**

```text
limited long-term data. Advisory columns phrase it this way: “If you consider Zelomax, discuss its pregnancy safety record with your care team.” Some reviewers highlight that patients appreciate a
```

### Subquestion 3

**Search text:** **Drug Interactions and Comorbidities**  

**Result 1**

```text
once-daily dosing, minimal drug interactions, and a favorable safety profile, Xenthera™ empowers your patients to breathe easier and reclaim their everyday routines.
```

**Result 2**

```text
👉 Learn more: #BreatheBoldly #XentheraJourney #ChronicNoMore
```

### Your observations 3

Which part of the original question does each passage help answer?
Which parts remain unsupported? Retrieval always returns the nearest matches here,
even when the document does not contain an answer. A complete RAG pipeline would
next combine the relevant evidence and identify any gaps when generating its answer.

**Your response:**

The queries are broken down into essentially the benefits of Xenthera, contradictions to avoid and the interaction of drugs when in use. It seems as if the decomposition covers the entire scope of the original question. It may have perhaps not been as comprehensive as originally intended however.

Check that the subquestions cover benefits, contraindications, and drug interactions
or comorbidities. They should preserve the original question's scope. The retrieved
passages may still leave some parts unanswered.

## G. HyDE (Hypothetical Document Embeddings)

**The idea:** Generate a hypothetical answer and use its embedding to search for real passages. The generated text may resemble an answer-containing passage more closely than the original question does.

**How it works:**
1. Take the user's question
2. Ask an LLM to generate a hypothetical document that would answer it
3. Embed this hypothetical document (not the question)
4. Search for real documents similar to the hypothetical one

**Example transformation:**
- **Original question**: "What are the contraindications for Zelomax?"
- **Hypothetical document**: "Zelomax is contraindicated in patients with severe liver dysfunction, known hypersensitivity to the active ingredients, and during pregnancy, particularly between gestational days 6-19. Clinical studies have shown significant risks during the critical organogenesis phase. Patients with these conditions should not be prescribed Zelomax under any circumstances..."

**Why this can help:**
- **Document-to-document matching** can be more accurate than question-to-document
- Questions and answers have different semantic structures
- The hypothetical document uses terminology similar to real documents
- Embeddings can at times capture the "answer space" better than the "question space"

**Visual representation:**
```
Traditional RAG:
  "What are contraindications?" [question embedding]
     → Searches for documents based on question
     → [semantic mismatch]

HyDE:
  "What are contraindications?"
     → Generate: "Contraindications include severe liver dysfunction..." [document embedding]
     → Searches for documents based on hypothetical document embedding
     → [compare the retrieved passages]
```

**When to use:**
- Exploratory or conceptual questions
- When queries are open-ended
- When document-to-document similarity is more reliable than question-to-document

In the cell below, the user's question is stored in `example_query_hyde`. The prompt sent to the LLM is constructed in `hyde_prompt`, which instructs the model to generate a hypothetical document excerpt that would answer the question, written in the style of formal pharmaceutical documentation. The remaining code calls the Chat Completions API to generate the hypothetical document, then compares retrieval results using both the original question and the hypothetical document to demonstrate the difference.

#### Run the Following Cell

In [19]:
# === Example: HyDE ===

example_query_hyde = "What is Xenthera used for?"

# Prompt for HyDE
hyde_prompt = f"""You are a pharmaceutical documentation expert. Given a question, generate a detailed hypothetical document excerpt that would answer it.

Write as if you are creating content from a medical document with:
- Professional medical terminology
- Specific details and facts
- Formal documentation style

Question: {example_query_hyde}

Hypothetical document excerpt:"""

# Call OpenAI API to generate hypothetical document
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": hyde_prompt}],
)

hypothetical_doc = response.choices[0].message.content.strip()

# Retrieve using hypothetical document
hyde_results = retriever.invoke(hypothetical_doc)

# Compare with baseline
baseline_results = retriever.invoke(example_query_hyde)

output = f"""
## HyDE Example

**Original question:**
"{example_query_hyde}"

**Hypothetical document generated:**
```
{hypothetical_doc}
```

---

### Retrieval Comparison

**Baseline (using question directly):**
```
{baseline_results[0].page_content}...
```

---

**HyDE (using hypothetical document):**
```
{hyde_results[0].page_content}...
```

---
"""

display(Markdown(output))


## HyDE Example

**Original question:**
"What is Xenthera used for?"

**Hypothetical document generated:**
```
---

**Section 1.2: Therapeutic Indications of Xenthera**  

**Brand Name:** Xenthera  
**Generic Name:** Zirametilumab  
**Pharmacologic Category:** Monoclonal Antibody (anti-inflammatory, immunomodulator)  

**Indication and Usage:**  
Xenthera (zirametilumab) is indicated for the treatment of **moderate to severe rheumatoid arthritis (RA)** in adult patients who have demonstrated an inadequate response or intolerance to one or more disease-modifying antirheumatic drugs (DMARDs), including methotrexate. Xenthera is designed to modulate immune system pathways by selectively inhibiting pro-inflammatory cytokines, with the primary mechanism of action being targeted suppression of interleukin-6 (IL-6) signaling.  

In addition to its use in rheumatoid arthritis, Xenthera has been approved as a therapeutic agent for the following conditions:  

1. **Psoriatic Arthritis (PsA):**  
   Xenthera is approved for patients with active psoriatic arthritis to reduce joint inflammation, prevent structural joint damage, and improve physical function in cases where conventional therapies have been ineffective.  

2. **Systemic Lupus Erythematosus (SLE):**  
   As an adjunct therapy, Xenthera has shown efficacy in reducing disease activity in patients with moderate to severe SLE, particularly in those with refractory disease or glucocorticoid dependence.  

3. **Ankylosing Spondylitis (AS):**  
   Xenthera may be used in the treatment of patients with active ankylosing spondylitis to improve spinal mobility, relieve pain, and control inflammation in cases unresponsive to nonsteroidal anti-inflammatory drugs (NSAIDs).  

**Mechanism of Action:**  
Zirametilumab binds with high specificity to the IL-6 receptor (IL-6R), inhibiting both soluble and membrane-bound IL-6R-mediated activation. This prevents downstream Janus kinase-signal transducer and activator of transcription (JAK-STAT) pathway signaling, thereby reducing inflammation and immune-mediated tissue damage. Sustained IL-6 inhibition has been associated with the modulation of acute-phase reactants such as C-reactive protein (CRP), which is a key biomarker of systemic inflammation.  

**Clinical Data:**  
Efficacy and safety of Xenthera have been confirmed in pivotal Phase 3 clinical trials (e.g., the XTRA Study and SYNTH-2 Study) involving over 2,500 adult patients with autoimmune and inflammatory conditions. Trial outcomes reflect a statistically significant improvement in the American College of Rheumatology (ACR20/50/70) response criteria after 24 weeks of treatment, as well as notable reductions in baseline Disease Activity Score-28 (DAS28) levels.  

**Administration:**  
Xenthera is administered as a subcutaneous injection, with an initial loading dose followed by maintenance therapy tailored to the patient’s clinical response and tolerability. The recommended dosing schedule is 200 mg every four weeks for rheumatoid arthritis and may vary by indication as per the prescribing information.  

**Contraindications and Limitations of Use:**  
Xenthera is contraindicated in patients:  
- With a history of severe hypersensitivity to zirametilumab or any of its excipients.  
- With active, serious infections, including tuberculosis.  
- Receiving live vaccines concurrently with Xenthera therapy.  

It has not been established whether Xenthera is safe or effective in pediatric populations under 18 years of age.  

For a comprehensive review of Xenthera’s safety profile, please refer to Section 4.8, "Adverse Reactions," and Section 5, "Warnings and Precautions," of this prescribing information.  

--- 

**End of Excerpt**
```

---

### Retrieval Comparison

**Baseline (using question directly):**
```
Introducing Xenthera™ – The Next Generation in Chronic Respiratory Care...
```

---

**HyDE (using hypothetical document):**
```
Help your patients move beyond the flare-up cycle. Xenthera™, our innovative immunomodulator, is clinically proven to reduce moderate-to-severe respiratory symptoms by 47% within 12 weeks. With...
```

---


### Compare all retrieved passages for HyDE

The previous cell displays only the first result from each search. Here we will
display both results, using the searches that have already run. A useful passage
could appear second, so reading only the first result could change our conclusion.

**The hypothetical document is a search input, not evidence.** The model generated
it without reading our source document. It may invent claims or details that lead
the search in the wrong direction. Judge the results using the actual retrieved
passages and the original question.

In [20]:
# Reuse the existing results. This cell makes no additional API calls.
comparisons = [
    ("Original question: retrieved passages", baseline_results),
    ("Hypothetical document: retrieved passages", hyde_results),
]

for label, results in comparisons:
    display(Markdown(f"### {label}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Original question: retrieved passages

**Result 1**

```text
Introducing Xenthera™ – The Next Generation in Chronic Respiratory Care
```

**Result 2**

```text
With Xenthera™, you can break free from the pattern of flare-ups and fatigue. Backed by leading pulmonologists and real-world patient success, Xenthera™ helps reduce inflammation at the source —
```

### Hypothetical document: retrieved passages

**Result 1**

```text
Help your patients move beyond the flare-up cycle. Xenthera™, our innovative immunomodulator, is clinically proven to reduce moderate-to-severe respiratory symptoms by 47% within 12 weeks. With
```

**Result 2**

```text
With Xenthera™, you can break free from the pattern of flare-ups and fatigue. Backed by leading pulmonologists and real-world patient success, Xenthera™ helps reduce inflammation at the source —
```

**Discuss:** Did HyDE retrieve more useful evidence, repeat the same passages,
or move the search away from the original question? Identify one claim in the
hypothetical document and check whether the retrieved text actually supports it.

HyDE uses the generated document as the search input. Its wording may help locate
relevant passages, but invented details can also misdirect the search. Compare
the actual retrieved passages before deciding whether it helped.

### Your turn: HyDE

Your task is to implement HyDE by creating a prompt that generates hypothetical documents.

**Requirements:**
- Write a prompt that generates a hypothetical document answering the question
- The output should be a document excerpt (not just a rewritten question)
- Use professional, formal language as if from medical documentation
- Include specific details and terminology

**Key difference from query rewriting:**
- **Query rewriting**: Rephrases the question
- **HyDE**: Generates a hypothetical answer/document

Replace `[Write your instructions here]` in the prompt below with your own instructions.
Keep the query variable inside the braces unchanged, then run the cell and its retrieval comparison.

**Hints:**
- Ask the model to write as if creating documentation
- Emphasize professional medical language and specific details

#### Write your prompt, then run the following cell

In [21]:

query_hyde = "What are the safety concerns with Zelomax during pregnancy?"

# Prompt for HyDE

# BEGIN SOLUTION
hyde_prompt = f"""
You are a medical professional writing a hypothetical document. Ensure these requirements are met:
- The output should be a document excerpt (not just a rewritten question)
- Use professional, formal language as if from medical documentation
- Include specific details and terminology

Question: {query_hyde}

"""
# END SOLUTION

# Call OpenAI API to generate hypothetical document
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": hyde_prompt}]
)

hypothetical_doc = response.choices[0].message.content.strip()

# Test retrieval with both approaches
hyde_retrieval = retriever.invoke(hypothetical_doc)
baseline_retrieval = retriever.invoke(query_hyde)

output = f"""
## Your HyDE Implementation Test

**Original question:**
"{query_hyde}"

**Your hypothetical document:**
```
{hypothetical_doc}
```

---

### Retrieval Comparison

**Baseline (question):**
```
{baseline_retrieval[0].page_content}...
```

**HyDE (hypothetical document):**
```
{hyde_retrieval[0].page_content}...
```

---
"""

display(Markdown(output))


## Your HyDE Implementation Test

**Original question:**
"What are the safety concerns with Zelomax during pregnancy?"

**Your hypothetical document:**
```
**Excerpt from Medical Documentation: Safety Profile of Zelomax (Hypothetical Drug) in Pregnant Populations**

**Background and Overview**:  
Zelomax, a pharmaceutical agent classified as a novel [insert drug class here], is indicated for the management of [insert condition(s) here]. While its efficacy in targeted populations has been well-documented, its safety profile during pregnancy requires careful scrutiny due to the potential implications for both maternal and fetal health.

**Teratogenicity Concerns**:  
Preclinical animal studies have demonstrated dose-dependent teratogenic effects when Zelomax was administered during organogenesis. These effects included structural malformations such as craniofacial anomalies, neural tube defects, and cardiac septal abnormalities. It is important to note that these findings have not been conclusively observed in human cohorts, as limited data exist from pregnant individuals.

**Placental Transfer and Fetal Exposure**:  
Pharmacokinetic studies suggest that Zelomax readily crosses the placenta, achieving fetal plasma concentrations comparable to maternal levels. This raises concerns regarding potential interference with early embryonic development and long-term neurocognitive outcomes in exposed offspring.

**Pregnancy Category and Classification**:  
As per the FDA's Pregnancy and Lactation Labeling Rule (PLLR), Zelomax falls under Category [C/D - tailored based on details known], indicating that the potential risks to fetal development may outweigh the benefits of treatment in certain cases. Clinical use during pregnancy is advised only if the therapeutic benefit justifies the potential risk to the fetus.

**Reported Adverse Outcomes**:  
In post-marketing surveillance, a small number of case reports have documented adverse pregnancy outcomes, including preterm labor, intrauterine growth restriction (IUGR), and spontaneous pregnancy loss among individuals exposed to Zelomax during the first trimester. Confounding variables, such as coexisting maternal comorbidities, should be considered when interpreting these findings.

**Recommendations for Clinical Practice**:  
1. **Avoid Use During First Trimester**: Zelomax should generally be avoided during the first trimester unless no suitable alternatives exist and the maternal condition poses significant risk if left untreated.
   
2. **Preconception Counseling**: Females of reproductive potential should be advised to use reliable contraception during treatment with Zelomax and for an appropriate duration following the final dose to mitigate unintended fetal exposure.

3. **Risk-Benefit Assessment**: Decisions regarding the use of Zelomax in pregnant patients should involve a multidisciplinary approach, taking into account the severity of the maternal condition, availability of alternative therapies, and thorough risk-benefit discussions with the patient.

4. **Fetal Monitoring**: If Zelomax administration during pregnancy is deemed necessary, enhanced fetal monitoring should be implemented to detect potential signs of impaired development.

**Conclusion**:  
The use of Zelomax during pregnancy is associated with safety concerns that warrant significant caution. Rigorous clinical judgment and patient-specific risk assessments are essential in determining its appropriateness in pregnant individuals. Further research, including long-term epidemiological studies and controlled trials, is necessary to better delineate its safety profile and provide clearer guidance for prescribers.

- Signed,  
[Medical Professional's Name]  
[Credentials and Title]
```

---

### Retrieval Comparison

**Baseline (question):**
```
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy....
```

**HyDE (hypothetical document):**
```
Clinical newsletters routinely review the safety profile of Zelomax. Several case summaries note that, when prescribed during pregnancy, clinicians carefully weigh perceived safety benefits against...
```

---


### Compare all retrieved passages for HyDE

The previous cell displays only the first result from each search. Here we will
display both results, using the searches that have already run. A useful passage
could appear second, so reading only the first result could change our conclusion.

**The hypothetical document is a search input, not evidence.** The model generated
it without reading our source document. It may invent claims or details that lead
the search in the wrong direction. Judge the results using the actual retrieved
passages and the original question.

In [22]:
# Reuse the existing results. This cell makes no additional API calls.
comparisons = [
    ("Original question: retrieved passages", baseline_retrieval),
    ("Hypothetical document: retrieved passages", hyde_retrieval),
]

for label, results in comparisons:
    display(Markdown(f"### {label}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Original question: retrieved passages

**Result 1**

```text
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

**Result 2**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

### Hypothetical document: retrieved passages

**Result 1**

```text
Clinical newsletters routinely review the safety profile of Zelomax. Several case summaries note that, when prescribed during pregnancy, clinicians carefully weigh perceived safety benefits against
```

**Result 2**

```text
“safe” or “not so safe” it felt for them personally in each trimester of pregnancy. Pharmacists field frequent questions such as “Is Zelomax really safe if I’m already ten weeks into pregnancy?”
```

### Your observations 4

Did HyDE retrieve more useful evidence, repeat the same passages,
or move the search away from the original question? Identify one claim in the
hypothetical document and check whether the retrieved text actually supports it.

**Your response:**
It seemed as if the retrieval repeated some passaged but also did provide more useful evidence on its on. This result: Clinical newsletters routinely review the safety profile of Zelomax. Several case summaries note that, when prescribed during pregnancy, clinicians carefully weigh perceived safety benefits against was not seen before but now seems to be claimed and accurate under the new prompt.

The distinction to remember is that rewriting produces another question, while
HyDE produces a hypothetical answer. That generated answer is not evidence.
Check whether searching with it retrieved passages that support an answer to
the original question.

## H. Reflection

You have compared four ways to change the input to a retrieval system.

### What You Learned

**Key concepts:**
- **Query Rewriting:** Expressing the same intent in multiple alternative phrasings to increase retrieval coverage
- **Query Expansion:** Enriching queries with synonyms and related terms to bridge vocabulary gaps
- **Query Decomposition:** Breaking complex multi-part questions into simpler, focused sub-questions
- **HyDE (Hypothetical Document Embeddings):** Generating hypothetical answer documents for better semantic matching

**The core insight:**

User queries and documents can often live in different linguistic spaces.

Query transformation techniques bridge this gap by:
- Rephrasing questions to match document language
- Expanding vocabulary to capture domain-specific terminology
- Simplifying complex queries into manageable pieces
- Generating document-style text for better similarity matching

### Optional Discussion Questions

Your written responses belong in the four **Your observations** areas above.
The questions below are for discussion; additional written answers are not required.

1. **Technique selection:** Which transformation technique would work best for different query types? When would you choose query rewriting over HyDE?

2. **Combining techniques:** Could you use multiple transformation techniques together? For example, decompose a complex query, then expand each sub-question with synonyms?

3. **Domain adaptation:** How would you modify these prompts for different domains beyond pharmaceutical? What would change for legal documents, technical manuals, or financial reports?

4. **Trade-offs:** Each technique adds an extra LLM call before retrieval. What evidence would justify the added latency and cost?

5. **Retrieval strategy:** After transforming a query into multiple versions (e.g., 3 rewrites), do you retrieve separately and merge results, or combine the queries first?

6. **Failure modes:** What happens when query transformation goes wrong? How might a poorly decomposed question or an inaccurate hypothetical document hurt retrieval?

7. **Evaluation:** How would you measure whether query transformation is actually improving your RAG system's performance?

Keep these questions in mind as you continue learning about advanced RAG techniques and system design.

## Save Your Work

Save the notebook with your prompts, observations, and code outputs. Before finishing,
check that your explanations refer to the results you actually obtained.